# IAEDU Scratchbook

This notebook is for experimenting with the IAEDU agent endpoint while keeping a `ChatGpt.chat()`-style interface that returns one final string instead of token-by-token output.

Current finding from a live probe on 2026-04-07:
- `.../stream` works and returns `text/event-stream`.
- Removing `/stream` from this endpoint returns HTTP `404` with `Resource not found`.
- The stream emits `start`, `token`, `message`, and `done` events. The final answer appears in the `message` event, so we can collapse the stream into a normal string return.

In [1]:
import json
import os
import secrets
from pprint import pprint
from typing import Any, Dict, List, Optional

import requests

/opt/conda/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.1.0) or chardet (6.0.0.post1)/charset_normalizer (2.0.4) doesn't match a supported version!
  warnings.warn(


In [2]:
ENDPOINT_STREAM = "https://api.iaedu.pt/agent-chat//api/v1/agent/cmamvd3n40000c801qeacoad2/stream"
ENDPOINT_BASE = ENDPOINT_STREAM.removesuffix("/stream")
CHANNEL_ID = "cmj1i57292iz9lq01goukbuuv"
API_KEY = os.getenv("IAEDU_API_KEY", "")

DEFAULT_USER_INFO: Dict[str, Any] = {}
DEFAULT_USER_CONTEXT: Dict[str, Any] = {}

if not API_KEY:
    print("Set IAEDU_API_KEY in the environment before running request cells.")

print({
    "endpoint_stream": ENDPOINT_STREAM,
    "endpoint_base": ENDPOINT_BASE,
    "channel_id": CHANNEL_ID,
    "api_key_present": bool(API_KEY),
})

{'endpoint_stream': 'https://api.iaedu.pt/agent-chat//api/v1/agent/cmamvd3n40000c801qeacoad2/stream', 'endpoint_base': 'https://api.iaedu.pt/agent-chat//api/v1/agent/cmamvd3n40000c801qeacoad2', 'channel_id': 'cmj1i57292iz9lq01goukbuuv', 'api_key_present': True}


## IAEDU Adapter

This adapter intentionally mirrors the existing `chat(instructions, input) -> str` contract.

Important implementation detail: for multipart requests, do **not** set the `Content-Type` header manually. `requests` will add the correct multipart boundary when `files=` is used.

In [7]:
class IaEduCompatClient:

    """Small adapter that collapses the IAEDU event stream into a final string."""



    def __init__(

        self,

        endpoint: str,

        api_key: str,

        channel_id: str,

        timeout: int = 60,

    ) -> None:

        self.endpoint = endpoint.rstrip("/")

        self.api_key = api_key

        self.channel_id = channel_id

        self.timeout = timeout

        self.last_thread_id: Optional[str] = None

        self.last_events: List[Dict[str, Any]] = []



    def new_thread_id(self) -> str:

        """Create a URL-safe thread id for a new conversation."""

        return secrets.token_urlsafe(16)



    def _build_message(self, instructions: str, input_text: str) -> str:

        """Combine instructions and user input into the single IAEDU message field."""

        instructions = instructions.strip()

        input_text = input_text.strip()



        if instructions and input_text:

            return f"{instructions}\n\n{input_text}"

        return instructions or input_text



    def _build_files(

        self,

        instructions: str,

        input_text: str,

        thread_id: str,

        user_id: Optional[str] = None,

        user_info: Optional[Dict[str, Any]] = None,

        user_context: Optional[Dict[str, Any]] = None,

    ) -> Dict[str, tuple[None, str]]:

        """Build the multipart form payload expected by the IAEDU endpoint."""

        form_values = {

            "channel_id": self.channel_id,

            "thread_id": thread_id,

            "user_info": json.dumps(user_info or DEFAULT_USER_INFO),

            "message": self._build_message(instructions, input_text),

        }



        merged_context = user_context or DEFAULT_USER_CONTEXT

        if merged_context:

            form_values["user_context"] = json.dumps(merged_context)



        if user_id:

            form_values["user_id"] = user_id



        return {key: (None, str(value)) for key, value in form_values.items()}



    def _coerce_text(self, value: Any) -> str:

        """Extract text from string or nested JSON payloads."""

        if value is None:

            return ""



        if isinstance(value, str):

            return value



        if isinstance(value, list):

            return "".join(self._coerce_text(item) for item in value)



        if isinstance(value, dict):

            for key in ("content", "text", "message", "delta", "answer"):

                if key in value:

                    text = self._coerce_text(value[key])

                    if text:

                        return text



        return ""



    def _parse_event_line(self, line: str) -> Optional[Dict[str, Any]]:

        """Parse a single line from the IAEDU stream."""

        cleaned = line.strip()

        if not cleaned:

            return None



        if cleaned.startswith("data:"):

            cleaned = cleaned[5:].strip()



        if cleaned in {"[DONE]", "DONE", "done"}:

            return {"type": "done", "content": cleaned}



        try:

            payload = json.loads(cleaned)

        except json.JSONDecodeError:

            return {"type": "raw", "content": cleaned}



        if isinstance(payload, dict):

            return payload



        return {"type": "raw", "content": payload}



    def chat(

        self,

        instructions: str,

        input: str,

        thread_id: Optional[str] = None,

        user_id: Optional[str] = None,

        user_info: Optional[Dict[str, Any]] = None,

        user_context: Optional[Dict[str, Any]] = None,

    ) -> str:

        """Send a request and return one final merged answer string."""

        if not self.api_key:

            raise ValueError("Missing IAEDU API key. Set IAEDU_API_KEY first.")



        thread_id = thread_id or self.new_thread_id()

        self.last_thread_id = thread_id



        headers = {

            "x-api-key": self.api_key,

            "Accept": "text/event-stream, application/json, text/plain",

        }



        files = self._build_files(

            instructions=instructions,

            input_text=input,

            thread_id=thread_id,

            user_id=user_id,

            user_info=user_info,

            user_context=user_context,

        )



        token_buffer: List[str] = []

        final_message = ""

        error_message = ""

        events: List[Dict[str, Any]] = []



        with requests.post(

            self.endpoint,

            headers=headers,

            files=files,

            stream=True,

            timeout=self.timeout,

        ) as response:

            response.raise_for_status()



            for raw_line in response.iter_lines(decode_unicode=True):

                if raw_line is None:

                    continue



                event = self._parse_event_line(raw_line)

                if not event:

                    continue



                events.append(event)

                event_type = event.get("type", "")

                content = event.get("content")



                if event_type == "token":

                    token_buffer.append(self._coerce_text(content))

                elif event_type == "message":

                    final_message = self._coerce_text(content)

                elif event_type == "error":

                    error_message = self._coerce_text(content)



        self.last_events = events



        if error_message and not final_message and not token_buffer:

            raise RuntimeError(f"IAEDU error: {error_message}")



        text = final_message or "".join(token_buffer)

        return text.strip()



    def probe_endpoint(

        self,

        endpoint: str,

        message: str = "Reply with exactly: IAEDU_OK",

    ) -> Dict[str, Any]:

        """Send a one-off probe request and return status, headers, and body preview."""

        if not self.api_key:

            raise ValueError("Missing IAEDU API key. Set IAEDU_API_KEY first.")



        headers = {

            "x-api-key": self.api_key,

            "Accept": "text/event-stream, application/json, text/plain",

        }



        files = self._build_files(

            instructions="You are a concise assistant.",

            input_text=message,

            thread_id=self.new_thread_id(),

        )



        response = requests.post(

            endpoint,

            headers=headers,

            files=files,

            timeout=self.timeout,

        )



        return {

            "status_code": response.status_code,

            "content_type": response.headers.get("content-type", ""),

            "body_preview": response.text[:1000],

        }


In [8]:
agent = IaEduCompatClient(
    endpoint=ENDPOINT_STREAM,
    api_key=API_KEY,
    channel_id=CHANNEL_ID,
    timeout=60,
)

agent

## Smoke Test

This should print only `IAEDU_OK` if the stream parser is working correctly.

In [9]:
reply = agent.chat(
    instructions="You are a concise assistant.",
    input="Reply with exactly: IAEDU_OK",
)

print(reply)
print({"thread_id": agent.last_thread_id, "events_seen": len(agent.last_events)})

IAEDU_OK
{'thread_id': 'mJFHnffzELcbrJ3iX_iw8g', 'events_seen': 8}


In [6]:
agent.last_events[-5:]

[{'run_id': 'b40b980d-b7e7-400d-8b5d-91f4f3e35e0d',
  'type': 'start',
  'content': 'Processing'},
 {'run_id': 'b40b980d-b7e7-400d-8b5d-91f4f3e35e0d',
  'type': 'error',
  'content': 'Unexpected processing error'},
 {'run_id': 'b40b980d-b7e7-400d-8b5d-91f4f3e35e0d',
  'type': 'done',
  'content': 'b40b980d-b7e7-400d-8b5d-91f4f3e35e0d',
  'messageId': 'None'},
 {'run_id': 'b40b980d-b7e7-400d-8b5d-91f4f3e35e0d',
  'type': 'done',
  'content': 'b40b980d-b7e7-400d-8b5d-91f4f3e35e0d',
  'messageId': 'None'}]

## Probe `/stream` vs Base Endpoint

This checks whether removing `/stream` returns a normal final response. At the time of creation, the live probe returned `404` for the base endpoint.

In [10]:
stream_probe = agent.probe_endpoint(ENDPOINT_STREAM)
base_probe = agent.probe_endpoint(ENDPOINT_BASE)

pprint({
    "stream_probe": stream_probe,
    "base_probe": base_probe,
})

{'base_probe': {'body_preview': '{ "statusCode": 404, "message": "Resource not '
                                'found" }',
                'content_type': 'application/json',
                'status_code': 404},
 'stream_probe': {'body_preview': '{"run_id": '
                                  '"c6c29091-e70d-4f24-826b-c8ad7ec98dcd", '
                                  '"type": "start", "content": "Processing"}\n'
                                  '\n'
                                  '{"run_id": '
                                  '"c6c29091-e70d-4f24-826b-c8ad7ec98dcd", '
                                  '"type": "token", "content": "IA"}\n'
                                  '\n'
                                  '{"run_id": '
                                  '"c6c29091-e70d-4f24-826b-c8ad7ec98dcd", '
                                  '"type": "token", "content": "ED"}\n'
                                  '\n'
                                  '{"run_id": '
                      

## Ontology-Style Prompt Experiment

This keeps the same `chat()` shape used by the current codebase, so you can try prompt variants without dealing with stream tokens directly.

In [11]:
ontology_prompt = """
Generate a JSON array with 10 broad ontology classes for the Star Trek domain.
Return only valid JSON.
"""

ontology_reply = agent.chat(
    instructions=(
        "You are a taxonomy and ontology expert. "
        "Provide concise and accurate responses based on the user's queries."
    ),
    input=ontology_prompt,
)

print(ontology_reply)

```json
[
  {
    "class": "Species",
    "description": "Different sentient and non-sentient species in the Star Trek universe."
  },
  {
    "class": "Starships",
    "description": "Various types of spacecraft used by different factions and species."
  },
  {
    "class": "Planets",
    "description": "Planets and celestial bodies featured in the Star Trek universe."
  },
  {
    "class": "Factions",
    "description": "Political and military organizations, such as the Federation and Klingon Empire."
  },
  {
    "class": "Technology",
    "description": "Advanced technologies, including warp drives, transporters, and replicators."
  },
  {
    "class": "Characters",
    "description": "Key individuals and their roles in the Star Trek narrative."
  },
  {
    "class": "Starbases",
    "description": "Space stations and facilities used for operations and logistics."
  },
  {
    "class": "Events",
    "description": "Major historical and narrative events in the Star Trek timeline."
 

In [12]:
cleaned_reply = ontology_reply.strip()



if cleaned_reply.startswith("```"):

    cleaned_reply = cleaned_reply.split("\n", 1)[1]

    if cleaned_reply.endswith("```"):

        cleaned_reply = cleaned_reply.rsplit("\n```", 1)[0]



parsed = json.loads(cleaned_reply)

print(type(parsed), len(parsed))

parsed[:3] if isinstance(parsed, list) else parsed


<class 'list'> 10


[{'class': 'Species',
  'description': 'Different sentient and non-sentient species in the Star Trek universe.'},
 {'class': 'Starships',
  'description': 'Various types of spacecraft used by different factions and species.'},
 {'class': 'Planets',
  'description': 'Planets and celestial bodies featured in the Star Trek universe.'}]